In [1]:
%pip install pandas plotly
%pip install --upgrade nbformat

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import plotly.express as px

df = pd.read_csv('112_final.csv', encoding='utf-8')
print("Loaded:", df.shape)

Loaded: (1970922, 10)


In [3]:
# Translation from LT TO ENG
df.rename(columns={
    'X': 'longitude',
    'Y': 'latitude',
    'aukstesnis_ivykio_tipas': 'higher_level_incident_type',
    'zemesnis_ivykio_tipas': 'lower_level_incident_type',
    'metai': 'year',
    'menuo': 'month',
    'ivykio_tipo_kodas': 'incident_type_code',
    'ivykio_id': 'incident_id'
}, inplace=True)
df.head()

,incident_type_code,incident_id,year,month,higher_level_incident_type,lower_level_incident_type,object_id,longitude,latitude,data_quality_flag
0,2.1,8a05b17d59f0,2022,3,BPC-GMP,GMP įvykis,1,23.282309,55.955301,ok
1,2.1,c9ffee4c15df,2022,3,BPC-GMP,GMP įvykis,2,23.953132,54.928555,ok
2,2.1,c349aa73f059,2022,3,BPC-GMP,GMP įvykis,3,22.324267,56.309919,ok
3,2.1,23b71b2f9a07,2022,3,BPC-GMP,GMP įvykis,4,24.145582,55.157485,ok
4,6.1.2,a15a6960d21f,2022,3,Policijos įvykiai,KET pažeidimas,5,21.193270,55.669982,ok


In [4]:
print(df['higher_level_incident_type'].value_counts())

higher_level_incident_type
Policijos įvykiai                  836362
BPC-GMP                            817428
GMP įvykiai                        193876
PGT įvykiai                         49101
Konsultacija                        28332
Aplinkosauga                        26482
BPC TRUKDANTIS                       8442
SPEC tarnybos                        5396
LAKD                                 2760
eCall                                1342
BPC-PGT                              1297
Testavimo pratybos                     58
Pagalba 112                            35
POLICIJOS pareigūnų korupcija *        10
116000                                  1
Name: count, dtype: int64


In [5]:
print(df.columns.tolist())
print(df.shape)
df.head()

['incident_type_code', 'incident_id', 'year', 'month', 'higher_level_incident_type', 'lower_level_incident_type', 'object_id', 'longitude', 'latitude', 'data_quality_flag']
(1970922, 10)


,incident_type_code,incident_id,year,month,higher_level_incident_type,lower_level_incident_type,object_id,longitude,latitude,data_quality_flag
0,2.1,8a05b17d59f0,2022,3,BPC-GMP,GMP įvykis,1,23.282309,55.955301,ok
1,2.1,c9ffee4c15df,2022,3,BPC-GMP,GMP įvykis,2,23.953132,54.928555,ok
2,2.1,c349aa73f059,2022,3,BPC-GMP,GMP įvykis,3,22.324267,56.309919,ok
3,2.1,23b71b2f9a07,2022,3,BPC-GMP,GMP įvykis,4,24.145582,55.157485,ok
4,6.1.2,a15a6960d21f,2022,3,Policijos įvykiai,KET pažeidimas,5,21.193270,55.669982,ok


In [6]:
import plotly.express as px

# Count events per higher-level type, sorted
higher_counts = df['higher_level_incident_type'].value_counts().reset_index()
lower_counts = df['lower_level_incident_type'].value_counts().reset_index()
higher_counts.columns = ['type', 'count']
lower_counts.columns = ['type', 'count']

fig = px.bar(
    higher_counts,
    x='count',
    y='type',
    orientation='h',                     # horizontal — easier to read long Lithuanian labels
    title='Events by higher-level type (2021–2023)',
    text='count'                         # show count at the end of each bar
)
fig.update_layout(
    yaxis=dict(categoryorder='total ascending'),  # biggest bar at the top
    height=500
)
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.show()

In [7]:
years = sorted(df.year.unique())
print(f'The unique years mentioned in the dataset: {[int(year) for year in years]}')


The unique years mentioned in the dataset: [2021, 2022, 2023]


In [8]:
import plotly.express as px

monthly = df.groupby(['year', 'month']).size().reset_index(name='count')
monthly['period'] = monthly['year'].astype(str) + '-' + monthly['month'].astype(str).str.zfill(2)
monthly = monthly.sort_values('period')

fig = px.line(
    monthly,
    x='period',
    y='count',
    title='Events per month (2021–2023)',
    markers=True
)
fig.update_layout(xaxis_tickangle=-45, height=500, width=1200)
fig.show()
print(df.columns.tolist())


['incident_type_code', 'incident_id', 'year', 'month', 'higher_level_incident_type', 'lower_level_incident_type', 'object_id', 'longitude', 'latitude', 'data_quality_flag']


1. The first data point (Sept 2021) is unusually low — about 72k vs. ~100k for normal months.
This is likely not a data quality issue — it's probably that September 2021 was the first month they started recording in this format. The data collection began partway through the month, so it has fewer events. This is normal for the first month of any dataset.

2. There's a dramatic dip in late 2022.
Looking carefully at the x-axis, the dip happens around October-November 2022, with November 2022 dropping to ~5,000 events and December 2022 (or November) being around ~57,000 — a partial recovery. By January 2023 it's back to normal at ~112k.

3. The "normal" baseline is ~100,000 events per month.
Most months hover between 95k and 110k. That's the actual operational baseline — anything dramatically below that is suspicious.
What likely happened in late 2022
I can't know for sure without context from whoever provided the data, but the shape of the dip is informative:

One month near-zero (~5k events)
The next month at about half-normal (~57k)
Then back to normal

This pattern is consistent with a system outage or migration — not a real-world drop in emergencies. Emergencies don't suddenly stop happening in Lithuania for a month. The most plausible explanations are:
    A database/dispatch system migration where data from that month was lost or moved elsewhere
    A change in how events were categorized (maybe records exist but use a different field that wasn't included in this export)
    A pipeline failure in whoever extracted this CSV

In [9]:
# --- Step 1: compute monthly counts and identify low-volume months ---
monthly_counts = df.groupby(['year', 'month']).size().reset_index(name='count')

# Use a threshold of 80,000 — anything below this is suspicious given the ~100k baseline
THRESHOLD = 80_000
bad_months = monthly_counts[monthly_counts['count'] < THRESHOLD]

print(f"Months flagged as incomplete (under {THRESHOLD:,} events):")
print(bad_months.to_string(index=False))
print(f"\nTotal flagged months: {len(bad_months)}")

# --- Step 2: build a set of (year, month) tuples for fast lookup ---
bad_set = set(zip(bad_months['year'], bad_months['month']))

# --- Step 3: add the flag column to df ---
df['data_quality_flag'] = df.apply(
    lambda row: 'incomplete' if (row['year'], row['month']) in bad_set else 'ok',
    axis=1
)

# --- Step 4: verify ---
print("\nFlag distribution:")
print(df['data_quality_flag'].value_counts())
print(f"\nFlagged rows as a percentage: {(df['data_quality_flag'] == 'incomplete').mean() * 100:.2f}%")

Months flagged as incomplete (under 80,000 events):
 year  month  count
 2021      9  72030
 2022     10   4168
 2022     11  57543

Total flagged months: 3

Flag distribution:
data_quality_flag
ok            1837181
incomplete     133741
Name: count, dtype: int64

Flagged rows as a percentage: 6.79%


In [10]:
import plotly.express as px

# Sample to keep things responsive
SAMPLE_SIZE = 30_000
sample = df.sample(n=SAMPLE_SIZE, random_state=42)

fig = px.scatter_map(
    sample,
    lat='latitude',
    lon='longitude',
    color='higher_level_incident_type',
    hover_data=['lower_level_incident_type', 'year', 'month'],
    zoom=6,
    height=700,
    title=f'Random sample of {SAMPLE_SIZE:,} events (out of {len(df):,})',
    opacity=0.5
)
fig.update_layout(map_style='open-street-map', margin=dict(l=0, r=0, t=40, b=0))
fig.show()

In [11]:
df.to_csv('112_final.csv', index=False, encoding='utf-8')
print("Saved 112_final.csv with", df.shape[0], "rows and columns:")
print(df.columns.tolist())

Saved 112_final.csv with 1970922 rows and columns:
['incident_type_code', 'incident_id', 'year', 'month', 'higher_level_incident_type', 'lower_level_incident_type', 'object_id', 'longitude', 'latitude', 'data_quality_flag']


In [12]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sample = df.sample(n=40_000, random_state=42)
top_types = df['higher_level_incident_type'].value_counts().head(4).index.tolist()

# 2x2 grid of map subplots
fig = make_subplots(
    rows=2, cols=2,
    specs=[[{"type": "scattermap"}, {"type": "scattermap"}],
           [{"type": "scattermap"}, {"type": "scattermap"}]],
    subplot_titles=top_types
)

# Add one map trace per type
positions = [(1, 1), (1, 2), (2, 1), (2, 2)]
for event_type, (r, c) in zip(top_types, positions):
    subset = sample[sample['higher_level_incident_type'] == event_type]
    fig.add_trace(
        go.Scattermap(
            lat=subset['latitude'],
            lon=subset['longitude'],
            mode='markers',
            marker=dict(size=4, opacity=0.4),
            name=event_type,
            showlegend=False
        ),
        row=r, col=c
    )

# Each map needs its own center/zoom set
for i in range(1, 5):
    fig.update_layout(**{
        f'map{i if i > 1 else ""}': dict(
            style='open-street-map',
            center=dict(lat=55.2, lon=23.9),  # roughly center of Lithuania
            zoom=5.5
        )
    })

fig.update_layout(height=800, title_text='Spatial distribution by event type (top 4)')
fig.show()

In [13]:
all_lower = df['lower_level_incident_type'].value_counts()
print(f"Total distinct lower-level types: {len(all_lower)}")
print()
print(all_lower.to_string())

Total distinct lower-level types: 154

lower_level_incident_type
GMP įvykis                                                              744798
KET pažeidimas                                                          189390
Įvairūs viešosios tvarkos pažeidimai                                    162397
Turtinė veika anksčiau                                                   87722
Smurtas artimoje aplinkoje                                               83206
Kompleksinis įvykis                                                      72584
Neklasifikuoti                                                           59263
Pavojus eismo saugumui                                                   44318
Nusikaltimai asmeniui dabar                                              33912
Turtinė veika dabar                                                      30826
Konsultacija                                                             28332
Lavonas                                                           

Clusters:

Confirmed bucket placements:

Neramumas / susijaudinimas → cross-bucket (public_order AND medical/psychiatric)
Kompleksinis įvykis → wildcard (50m radius)
Pavojus gyvybei → wildcard (50m radius)
Mirties konstatavimas → own bucket with Lavonas
Auto 1, Auto 2 → traffic_accident
Sprogimas ar sprogimo grėsmė → fire
Bandymas nusižudyti → medical/psychiatric (out of violence)
Įvykiai su laukiniais gyvūnais → single wildlife bucket (kept)
LAKD → traffic_* (road authority calls)
116000 → excluded (single record, can't cluster meaningfully)
Plus earlier: Žolė + Žolė (SA) + Narkotinės medžiagos → new drugs bucket

Standard buckets (refined from earlier draft):

medical_general (including specific symptoms — folded them in)
traffic_accident
traffic_violation
trauma
violence (minus suicide attempt)
public_order
property_crime
fire (plus explosions)
rescue (plus Mirties konstatavimas + Lavonas)
environmental
psychiatric (now includes suicide attempt + Neramumas shared with public_order)
wildlife
drugs
other_serious